I installed and imported the Pandas library.

In [5]:
!pip install pandas

In [6]:
import pandas as pd

I have uploaded all the Olist and Marketing Funnel tables to be used in the Nova7 scenario.

In [7]:
customers = pd.read_csv("../data/raw_data/olist/olist_customers_dataset.csv")
geo_locations = pd.read_csv("../data/raw_data/olist/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw_data/olist/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw_data/olist/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw_data/olist/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw_data/olist/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw_data/olist/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw_data/olist/olist_sellers_dataset.csv")
product_category_name_translations = pd.read_csv("../data/raw_data/olist/product_category_name_translation.csv")
closed_deals = pd.read_csv("../data/raw_data/marketing/olist_closed_deals_dataset.csv")
marketing_leads = pd.read_csv("../data/raw_data/marketing/olist_marketing_qualified_leads_dataset.csv")

In [8]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [9]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


The number of customer_unique_ids is less than the total number of customer_ids. This means there are multiple rows with some identical unique_ids.

The `customer_zip_code_prefix` field is stored as an int64, but this is a categorical/descriptive field, not a numerical measure. For example, aggregate statistics like mean/standard deviation are meaningless here. Therefore, I will convert it to a string during the cleanup phase.

In [10]:
customers.describe(include="all")

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
count,99441,99441,99441.000000,99441,99441
unique,99441,96096,NaN,4119,27
top,06b8999e2fba1a1fbc88172c00ba8bc7,8d50f5eadf50201ccdcedfb9e2ac8455,NaN,sao paulo,SP
freq,1,17,NaN,15540,41746
mean,NaN,NaN,35137.474583,NaN,NaN
std,NaN,NaN,29797.938996,NaN,NaN
min,NaN,NaN,1003.000000,NaN,NaN
25%,NaN,NaN,11347.000000,NaN,NaN
50%,NaN,NaN,24416.000000,NaN,NaN
75%,NaN,NaN,58900.000000,NaN,NaN


This code shows how many different rows (i.e., how many different customer_IDs) each customer_unique_id appears in the customers table. For example, the actual customer named 8d50f5eadf50201ccdcedfb9e2ac8455 is registered with 17 different customer_IDs in the table. A single customer appears to have multiple customer records. The underlying business reason is currently unknown and requires further investigation. For example, the same person may have placed orders using different email addresses, or a customer may have deleted their account and then registered again; such business rules may apply.

In [11]:
customers.customer_unique_id.value_counts().sort_values(ascending=False)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
ca77025e7201e3b30c44b472ff346268     7
6469f99c1f9dfae7733b25662e7f1782     7
                                    ..
26c602fff4586cb473c2c37abe87caef     1
99fefcc024154d6088c096ff42edb1cd     1
edf60415af3f7ec6da031ebbb8abb471     1
56c964ce504f3dce08f3f1df858eccd6     1
84732c5050c01db9b23e19ba39899398     1
Name: count, Length: 96096, dtype: int64

I answered the question "How many customer_unique_id instances occur more than once?" in the relevant sections.

In [12]:
repeat_counts = customers['customer_unique_id'].value_counts()
(repeat_counts > 1).sum()

np.int64(2997)

Approximately 3% of Nova7's total unique customers appear under multiple customer_IDs in the system. This indicates that we should use customer_unique_ID instead of customer_ID in "repeat customer" analytics; otherwise, we might mistakenly count the same person as multiple different customers.

In [13]:
repeat_rate = (repeat_counts > 1).sum() / customers['customer_unique_id'].nunique() * 100
repeat_rate

np.float64(3.1187562437562435)

In [14]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [15]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [16]:
orders.describe()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-03-31 15:08:21,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-14 20:02:44,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


Most of the orders have been delivered. However, there are 7 more cases besides those. I checked the numbers for each one.

In [17]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Some orders with blank delivery dates have been shipped, some have been cancelled, some are out of stock, etc. However, 8 orders show "delivered" but the "order_delivered_customer_date" section is empty. This is a data quality issue. There may be missing or incorrectly entered data.

In [18]:
orders[orders['order_delivered_customer_date'].isnull()]['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

I'm curious about the date range covered by the dataset. I might need this for time-based trend analysis. Here, I converted the string to datetime because it's more accurate to work with date columns as datetimes.

In [19]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_purchase_timestamp'].min(), orders['order_purchase_timestamp'].max()

(Timestamp('2016-09-04 21:15:19'), Timestamp('2018-10-17 17:30:18'))

I examined the number of orders per month. This part is important for analyzing how order volume progresses. Some months show a significant decrease compared to others. There may have been an interruption during the data collection period. If so, I will remove these periods from the trend analysis. It's important that we notice these things, otherwise misinterpretations can occur. 

In [20]:
orders['order_purchase_timestamp'].dt.to_period('M').value_counts().sort_index()

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

In [21]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [22]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


The table has 112,650 rows, more than the number of rows in the orders table. This is because an order can contain multiple products, and therefore order_items has more rows.

The order_item_id value is max=21. This means there are 21 different products in one order. This could be a bulk purchase.

In the price column, the standard deviation is even larger than the mean itself. This means the data isn't evenly distributed around the mean; it's spread over a wide range. A right-skewed distribution is present. Most products are cheap, but a small number of expensive products are pushing the mean and standard deviation upwards.

In such cases, the median is more reliable than the mean. When I create a visualization of the average value of orders in the future, I should not ignore these values.

In [23]:
order_items.describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


There are orders where the freight_value min = 0.00, meaning the shipping cost is zero. This might be a "free shipping" order, but I'm checking anyway.
There are 383 orders like this, and based on my dataset, it's a logical scenario.

In [24]:
(order_items['freight_value'] == 0).sum()

np.int64(383)

In [25]:
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [26]:
order_payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


The max=29 for `payment_sequential` is a noteworthy number. Different payment methods might have been used, but 29 doesn't seem very realistic. The system might have given an error, the customer might have tried repeatedly, the system might have split the payment, or it could be a data structure issue.

Again, there's a large difference between the maximum payout amount and the average, and the standard deviation remains above the average. As above, payouts are generally in the lower range, but some are higher than the average. I can confirm this from the 75% figure as well.

Order IDs can be repeated in this table. In fact, this table has more rows than the orders table because an order can be paid for in multiple ways, and a separate record is created for each payment method.

In [27]:
order_payments.describe()

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


There can be multiple reasons for a high `payment_sequential` value. To understand this, I examined the payment type in cases where there were more than 10 sequential payments. According to this output, a customer used multiple coupons in the same order, and the system recorded each coupon as a separate payment.

In [28]:
order_payments[order_payments['payment_sequential'] > 10]['payment_type'].value_counts()

payment_type
voucher    127
Name: count, dtype: int64

I wanted to examine the order with 29 payment records separately.

In [29]:
order_payments.groupby("order_id").size().sort_values(ascending=False)

order_id
fa65dad1b0e818e3ccc5cb0e39231352    29
ccf804e764ed5650cd8759557269dc13    26
285c2e15bebd4ac83635ccc563dc71f4    22
895ab968e7bb0d5659d16cd74cd1650c    21
fedcd9f7ccdc8cba3a18defedd1a5547    19
                                    ..
56bd45163229b35ca0ab490c1e3d3233     1
56bc98e6d5b88c2cdb905f2fbec2ca3a     1
56bbc7d92e6e74b8782abbf5ee336a92     1
56bafc014f8ed2f34cfe598592c65fd8     1
fffe41c64501cc87c801fd61db3f6244     1
Length: 99440, dtype: int64

As you can see, all records are of the voucher type. There are two 0.00 values, and the amounts are different from each other. These 0.00 values ​​could have several causes. It could be a cancelled voucher, a technical issue, or a rounding error.

In [30]:
order_payments[
    order_payments["order_id"] == "fa65dad1b0e818e3ccc5cb0e39231352"
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
4885,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
24879,fa65dad1b0e818e3ccc5cb0e39231352,25,voucher,1,3.68
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
32519,fa65dad1b0e818e3ccc5cb0e39231352,11,voucher,1,4.03


In [31]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [32]:
order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


Mostly rated 5 stars. However, this dataset only contains customers who submitted reviews, so it may not represent the satisfaction of all customers.

In [33]:
order_reviews.describe()

,review_score
count,99224.000000
mean,4.086421
std,1.347579
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


Review scores show a polarized (bimodal-leaning) pattern: 5-star is most common, but 1-star ranks third — more common than 2-3 star. This is a typical pattern in customer feedback, where neutral experiences are less likely to prompt a review than very positive or very negative ones.

In [34]:
order_reviews.review_score.value_counts()

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

The `order_reviews` table normally has 99,225 rows, but there are 98,673 different `order_id` in the table. This suggests that some orders have multiple reviews. This must be handled carefully during merging — a naive merge could duplicate order-level data. Will likely take the latest review per order, or aggregate review scores, during the cleaning phase.

In [35]:
order_reviews['order_id'].nunique()

98673

In [36]:
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])

Logically, the `review_answer_timestamp` should always come after the `review_creation_date`. This is because a reply to a comment comes after the comment is written. I tested this here, and it's consistent. If it weren't 0, there might be a data quality anomaly.

In [37]:
(order_reviews['review_answer_timestamp'] < order_reviews['review_creation_date']).sum()

np.int64(0)

I examined how many days it typically takes to respond to a review. The average is 2.5 days, but the standard deviation is very high (9.89) — this is driven by extreme outliers, with a maximum of 518 days. Since review_answer_timestamp has no nulls, every review does have a recorded response — but some responses took an unusually long time, possibly due to delayed processing or batch handling on the platform's side. Looking at the quartiles (25%: 1 day, 50%: 1 day, 75%: 3 days), the typical response time is much shorter than the mean suggests — again, median is more representative here than mean.

In [38]:
order_reviews['response_time_days'] = (order_reviews['review_answer_timestamp'] - order_reviews['review_creation_date']).dt.days
order_reviews['response_time_days'].describe()

count    99224.000000
mean         2.582248
std          9.890526
min          0.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        518.000000
Name: response_time_days, dtype: float64

In [39]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In the original dataset, the column name was misspelled as "lenght", it should have been "length".

In [40]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [41]:
products.describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


The minimum value entered for weight appears to be "0". I checked how many of these there are.

In [42]:
(products['product_weight_g'] == 0).sum()

np.int64(4)

I checked what these four products are. They all belong to the same category: "bed, table, bathroom". The weight field might have been overlooked during data entry, a standard template might have been used, or the system might have accepted 0 if it wasn't a required entry. I will correct this during data cleaning.

In [43]:
products[products['product_weight_g'] == 0]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


Some rows contain 610 null values, while others contain only 2. Using this code, I calculated the number of rows that are empty simultaneously in both the "product_category_name" and "product_weight_g" columns. Since there's only one, these two issues are likely unrelated. This simplifies the cleanup process.

In [44]:
products[products['product_category_name'].isnull()].shape[0]
products[products['product_weight_g'].isnull()].shape[0]
products[products['product_category_name'].isnull() & products['product_weight_g'].isnull()].shape[0]

1

In [45]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [46]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB


In [47]:
sellers.describe()

,seller_zip_code_prefix
count,3095.000000
mean,32291.059451
std,32713.453830
min,1001.000000
25%,7093.500000
50%,14940.000000
75%,64552.500000
max,99730.000000


This shows which states the sellers are concentrated in. The state of SP is dominant on both the buyer and seller sides.

In [48]:
sellers['seller_state'].value_counts()

seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
AC       1
PI       1
MA       1
AM       1
PA       1
Name: count, dtype: int64

sdr_id and sr_id; the IDs of the sales representatives who closed the deal, namely the Sales Development Rep and the Sales Rep.

won_date; the date the deal was closed/won.

business_segment, business_type; the seller's business type and sector.

has_company, has_gtin; fields such as "does the company have registration, does it have a product barcode system (GTIN)?".

declared_product_catalog_size, declared_monthly_revenue — the seller's self-declared catalog size and monthly revenue.

In [49]:
closed_deals.head()

,mql_id,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue
0,5420aad7fec3549a85876ba1c529bd84,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.0
1,a555fb36b9368110ede0f043dfc3b9a0,bbb7d7893a450660432ea6652310ebb7,09285259593c61296eef10c734121d5b,d3d1e91a157ea7f90548eef82f1955e3,2018-05-08 20:17:59,car_accessories,industry,eagle,NaN,NaN,NaN,reseller,NaN,0.0
2,327174d3648a2d047e8940d7d15204ca,612170e34b97004b3ba37eae81836b4c,b90f87164b5f8c2cfa5c8572834dbe3f,6565aa9ce3178a5caf6171827af3a9ba,2018-06-05 17:27:23,home_appliances,online_big,cat,NaN,NaN,NaN,reseller,NaN,0.0
3,f5fee8f7da74f4887f5bcae2bafb6dd6,21e1781e36faf92725dde4730a88ca0f,56bf83c4bb35763a51c2baab501b4c67,d3d1e91a157ea7f90548eef82f1955e3,2018-01-17 13:51:03,food_drink,online_small,NaN,NaN,NaN,NaN,reseller,NaN,0.0
4,ffe640179b554e295c167a2f6be528e0,ed8cb7b190ceb6067227478e48cf8dde,4b339f9567d060bcea4f5136b9f5949e,d3d1e91a157ea7f90548eef82f1955e3,2018-07-03 20:17:45,home_appliances,industry,wolf,NaN,NaN,NaN,manufacturer,NaN,0.0


The columns `has_company`, `has_gtin`, and `declared_product_catalog_size` contain many null values. This is a high number, so it's probably not a random omission. They could be optional form questions or fields added later and recently asked. This missing data issue needs to be addressed in the analytics section.

In [52]:
closed_deals.info()

<class 'pandas.DataFrame'>
RangeIndex: 842 entries, 0 to 841
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   mql_id                         842 non-null    str    
 1   seller_id                      842 non-null    str    
 2   sdr_id                         842 non-null    str    
 3   sr_id                          842 non-null    str    
 4   won_date                       842 non-null    str    
 5   business_segment               841 non-null    str    
 6   lead_type                      836 non-null    str    
 7   lead_behaviour_profile         665 non-null    str    
 8   has_company                    63 non-null     object 
 9   has_gtin                       64 non-null     object 
 10  average_stock                  66 non-null     str    
 11  business_type                  832 non-null    str    
 12  declared_product_catalog_size  69 non-null     float64
 13  d

declared_monthly_revenue is heavily skewed: median is 0, mean is ~73K, but max is 50M — a small number of large sellers pull the average far above the typical value. Zero may mean 'no revenue yet' (new sellers) or could be a placeholder for 'not declared' — this ambiguity will be noted, not resolved, since we can't confirm from the data alone.

In [53]:
closed_deals.describe()

,declared_product_catalog_size,declared_monthly_revenue
count,69.000000,8.420000e+02
mean,233.028986,7.337768e+04
std,352.380558,1.744799e+06
min,1.000000,0.000000e+00
25%,30.000000,0.000000e+00
50%,100.000000,0.000000e+00
75%,300.000000,0.000000e+00
max,2000.000000,5.000000e+07


In [54]:
marketing_leads.head()

,mql_id,first_contact_date,landing_page_id,origin
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search


`first_contact_date` is the date the lead first made contact, `landing_page_id` is the landing page they came from, and `origin` is the marketing source.

In [55]:
marketing_leads.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   mql_id              8000 non-null   str  
 1   first_contact_date  8000 non-null   str  
 2   landing_page_id     8000 non-null   str  
 3   origin              7940 non-null   str  
dtypes: str(4)
memory usage: 250.1 KB


In [56]:
marketing_leads.describe()

,mql_id,first_contact_date,landing_page_id,origin
count,8000,8000,8000,7940
unique,8000,336,495,10
top,dac32acd4db4c29c230538b72f8dd87d,2018-05-02,b76ef37428e6799c421989521c0e5077,organic_search
freq,1,93,912,2296


This checks if all mql_id's in closed_deals actually exist in the marketing_leads table. If it returns True, the relationship between the two tables is consistent, so I can confidently join them.

If it returns False, some closed_deals would be missing source lead records.

In [59]:
closed_deals['mql_id'].isin(marketing_leads['mql_id']).all()

np.True_

The marketing_leads table has 8000 rows, while the closed_deals table has 842 rows. This means some of them have converted to resellers. This allows me to calculate the conversion rate.

Approximately 1 out of every 10 leads captured by marketing turns into an actual salesperson.

In [60]:
conversion_rate = closed_deals['mql_id'].nunique() / marketing_leads['mql_id'].nunique() * 100
conversion_rate

10.525